# 🤖 Lab 5: AI for Data Science
## Lecture 22 companion — AI for Tabular Data + Time Series

This notebook accompanies **Lecture 22 (AI for Data Science)**.
It covers the two domains where data scientists work most:
**tabular data** and **time series**, and asks the same question for each:
*can modern foundation models challenge the long-reigning gold standards?*

### Part A — AI for Tabular Data
1. XGBoost / LightGBM recap & benchmark
2. TabPFN: a foundation model for tabular data
3. Performance vs dataset size — when does each win?
4. Text-to-SQL: natural-language access to tables

### Part B — AI for Time Series
5. Prophet: decomposition-based forecasting
6. Chronos: zero-shot probabilistic forecasting
7. Prophet vs Chronos — side-by-side
8. Practical guide: choosing the right approach

---

## 0. Environment Setup

In [ ]:
!pip install xgboost lightgbm tabpfn prophet chronos-forecasting scikit-learn pandas numpy matplotlib -q

In [ ]:
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import (
    load_breast_cancer, load_iris, load_wine, make_classification
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
print('🖥️ Environment ready')

---
# Part A: AI for Tabular Data

## 1. Why tabular data is challenging for deep learning

| Challenge | Why it hurts NNs |
|-----------|------------------|
| **Heterogeneous features** | Mixed types: categorical, numerical, ordinal — no spatial / sequential structure |
| **Small dataset sizes** | Many real-world tables have only hundreds to thousands of rows |
| **Feature interactions** | Tree splits capture irregular interactions naturally |
| **No universal pretraining** | Unlike vision (ImageNet) or NLP (text corpora), no standard tabular pretraining existed |

> **Grinsztajn et al. (NeurIPS 2022)**: trees still beat deep learning on typical tabular data.
> They are robust to uninformative features, preserve data orientation, and handle irregular targets.
> NNs tend to oversmooth and need more data.

## 2. XGBoost — Quick Recap

XGBoost = **gradient boosting + L1/L2 regularization + column subsampling**.

$$\text{Obj}^{(t)} = \sum_i L\!\left(y_i, \hat{y}_i^{(t-1)} + f_t(x_i)\right) + \Omega(f_t)$$

Loss on the prediction + regularization on tree complexity.

In [ ]:
import xgboost as xgb

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)

xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    eval_metric='logloss',
)
xgb_model.fit(X_tr, y_tr)
preds = xgb_model.predict(X_te)
print(f'🌲 XGBoost accuracy: {accuracy_score(y_te, preds):.4f}')

## 3. LightGBM — Faster, Leaf-Wise Splitting

| Feature | XGBoost | LightGBM |
|---------|---------|----------|
| Split strategy | Level-wise (BFS) | **Leaf-wise (best-first)** |
| Speed | Fast | **Faster (histogram-based)** |
| Memory | Higher | **Lower (binning)** |
| Categorical | Requires encoding | **Native support** |
| Overfitting risk | **Lower** | Higher on small data |

Key practical knob: **`num_leaves`** instead of `max_depth`.

In [ ]:
import lightgbm as lgb

train_set = lgb.Dataset(X_tr, label=y_tr)
params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'num_leaves': 31,         # leaf-wise control
    'learning_rate': 0.05,
    'feature_fraction': 0.8,  # column sampling
    'verbose': -1,
}
lgb_model = lgb.train(params, train_set, num_boost_round=100)
preds_proba = lgb_model.predict(X_te)
preds = (preds_proba > 0.5).astype(int)
print(f'🌲 LightGBM accuracy: {accuracy_score(y_te, preds):.4f}')

## 4. Benchmark: XGBoost vs LightGBM

A quick head-to-head on three classic sklearn datasets — both should hit >95%.

In [ ]:
datasets = {
    'Breast Cancer': load_breast_cancer(return_X_y=True),
    'Iris':          load_iris(return_X_y=True),
    'Wine':          load_wine(return_X_y=True),
}

xgb_accs, lgb_accs = [], []
for name, (X, y) in datasets.items():
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )
    xm = xgb.XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        eval_metric='mlogloss', verbosity=0,
    )
    xm.fit(X_tr, y_tr)
    xgb_accs.append(accuracy_score(y_te, xm.predict(X_te)))

    lm = lgb.LGBMClassifier(
        n_estimators=100, num_leaves=31, learning_rate=0.05, verbose=-1,
    )
    lm.fit(X_tr, y_tr)
    lgb_accs.append(accuracy_score(y_te, lm.predict(X_te)))

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(datasets)); w = 0.35
ax.bar(x - w/2, xgb_accs, w, label='XGBoost', color='#26C281', edgecolor='black')
ax.bar(x + w/2, lgb_accs, w, label='LightGBM', color='#3498DB', edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(list(datasets.keys()))
ax.set_ylabel('Accuracy'); ax.set_ylim(0.85, 1.02)
ax.set_title('XGBoost vs LightGBM — Real Benchmark Results', fontweight='bold')
for i, (xa, la) in enumerate(zip(xgb_accs, lgb_accs)):
    ax.text(i - w/2, xa + 0.005, f'{xa:.3f}', ha='center', fontweight='bold')
    ax.text(i + w/2, la + 0.005, f'{la:.3f}', ha='center', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

print('💡 Both achieve >95% accuracy across the board — tree-based models still dominate.')

## 5. TabPFN — Foundation Model for Tabular Data

**Hollmann et al. (ICLR 2023)** — Prior-Data Fitted Networks.

Key insight: **don't pretrain on real tabular data** (it differs across domains).
Instead, pretrain a Transformer on **millions of synthetic datasets** sampled from a flexible prior
(Structural Causal Models + Bayesian NNs + Gaussian Processes).

| Step | What happens |
|------|--------------|
| **1. Define prior** | A distribution P(D) over tabular datasets |
| **2. Pretrain** | A 12-layer Transformer on synthetic samples from P(D) |
| **3. Inference** | Feed real (X_train, y_train, X_test) → predict in a single forward pass |

→ **No gradient updates at test time. No hyperparameter tuning.**
Similar in spirit to in-context few-shot learning in GPT.

In [ ]:
from tabpfn import TabPFNClassifier

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)

# That's it — no hyperparameters to tune
tab_clf = TabPFNClassifier()
tab_clf.fit(X_tr, y_tr)
preds = tab_clf.predict(X_te)
probs = tab_clf.predict_proba(X_te)

print(f'🧠 TabPFN accuracy: {accuracy_score(y_te, preds):.4f}')
print(f'   predict_proba shape: {probs.shape}')

## 6. XGBoost vs TabPFN — Side-by-Side

Same data, same test set. Look at the **API surface** (and the timing).

In [ ]:
print('━' * 60)
print('XGBoost (8 hyperparameters):')
print('━' * 60)
xgb_clf = xgb.XGBClassifier(
    n_estimators=100, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    eval_metric='logloss', verbosity=0,
)
t0 = time.time(); xgb_clf.fit(X_tr, y_tr); xgb_time = time.time() - t0
xgb_acc = accuracy_score(y_te, xgb_clf.predict(X_te))
print(f'  Accuracy: {xgb_acc:.4f}    Train time: {xgb_time:.2f}s')

print('\n' + '━' * 60)
print('TabPFN (0 hyperparameters):')
print('━' * 60)
tab_clf = TabPFNClassifier()
t0 = time.time(); tab_clf.fit(X_tr, y_tr); fit_time = time.time() - t0
t0 = time.time(); tab_preds = tab_clf.predict(X_te); pred_time = time.time() - t0
tab_acc = accuracy_score(y_te, tab_preds)
print(f'  Accuracy: {tab_acc:.4f}    fit: {fit_time:.2f}s   predict: {pred_time:.2f}s')

print('\n💡 Same accuracy on small datasets — dramatically less effort.')

## 7. Performance vs Dataset Size

TabPFN excels on small data; tree models scale better. Where is the crossover point?
We subsample a 10K-row synthetic classification problem and sweep the training-set size.

In [ ]:
X_full, y_full = make_classification(
    n_samples=10000, n_features=20, n_informative=15,
    n_redundant=2, n_classes=2, random_state=SEED,
)
X_tr_full, X_te_full, y_tr_full, y_te_full = train_test_split(
    X_full, y_full, test_size=2000, random_state=SEED, stratify=y_full,
)

sizes = [50, 100, 200, 500, 1000, 3000]
xgb_curve, lgb_curve, tab_curve = [], [], []

for n in sizes:
    X_n, y_n = X_tr_full[:n], y_tr_full[:n]
    m = xgb.XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, eval_metric='logloss', verbosity=0)
    m.fit(X_n, y_n)
    xgb_curve.append(accuracy_score(y_te_full, m.predict(X_te_full)))

    m = lgb.LGBMClassifier(n_estimators=100, num_leaves=31, learning_rate=0.05, verbose=-1)
    m.fit(X_n, y_n)
    lgb_curve.append(accuracy_score(y_te_full, m.predict(X_te_full)))

    m = TabPFNClassifier()
    m.fit(X_n, y_n)
    tab_curve.append(accuracy_score(y_te_full, m.predict(X_te_full)))

    print(f'n={n:>5}: XGB={xgb_curve[-1]:.3f}  LGB={lgb_curve[-1]:.3f}  TabPFN={tab_curve[-1]:.3f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(sizes, xgb_curve, '-o', label='XGBoost (tuned)', color='#26C281', linewidth=2, markersize=8)
ax.plot(sizes, lgb_curve, '-s', label='LightGBM (tuned)', color='#7B5BE0', linewidth=2, markersize=8)
ax.plot(sizes, tab_curve, '-^', label='TabPFN (zero-shot)', color='#26B7DB', linewidth=2, markersize=8)
ax.axvline(x=3000, color='orange', linestyle='--', alpha=0.6, label='TabPFN v1 limit (~3K)')
ax.set_xscale('log')
ax.set_xlabel('Training set size'); ax.set_ylabel('Accuracy')
ax.set_title('Performance vs Dataset Size: TabPFN vs Tree Models', fontweight='bold')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print('\n💡 TabPFN starts strong on tiny data; tree models catch up as n grows.')

## 8. Text-to-SQL — Natural-Language Access to Tables

Tabular data is most useful when **anyone** can query it.
An LLM can translate plain English into SQL, execute it, and return the answer.

### Pattern (LangChain + an LLM):
```python
from langchain_community.utilities import SQLDatabase
from langchain_openai import ChatOpenAI
from langchain.chains import create_sql_query_chain

db = SQLDatabase.from_uri('sqlite:///company.db')
llm = ChatOpenAI(model='gpt-4', temperature=0)
chain = create_sql_query_chain(llm, db)

query = chain.invoke({'question': 'Top 5 employees by salary in engineering'})
result = db.run(query)
```

Spider benchmark SOTA is ~91% (GPT-4 + agent). Two approaches exist:
- **Text-to-SQL**: NL → SQL → DB → result
- **Table QA**: serialize table → LLM reasons → answer directly (TAPAS, Chain-of-Table)

Below we build a tiny in-memory database and demonstrate the execution side
(without consuming an API key).

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE employees (
    id INTEGER PRIMARY KEY,
    name TEXT,
    dept TEXT,
    salary INTEGER
);
INSERT INTO employees VALUES
    (1, 'Alice',   'engineering', 95000),
    (2, 'Bob',     'engineering', 110000),
    (3, 'Charlie', 'engineering', 88000),
    (4, 'Diana',   'sales',       72000),
    (5, 'Eve',     'sales',       82000),
    (6, 'Frank',   'engineering', 130000),
    (7, 'Grace',   'marketing',   78000);
''')
conn.commit()

# Imagine the LLM produced this SQL from the question:
# Q: 'Top 5 employees by salary in engineering'
sql = """
SELECT name, salary FROM employees
WHERE dept = 'engineering'
ORDER BY salary DESC
LIMIT 5;
"""

print('🔮 Q: "Top 5 employees by salary in engineering"')
print(f'💻 SQL:{sql}')
print('📈 Result:')
for row in cur.execute(sql):
    print(f'   {row}')

---
# Part B: AI for Time Series

Time series shows up everywhere: finance, energy, retail, healthcare, climate, IoT.
Forecasting evolved through four eras:

| Era | Models |
|-----|--------|
| Classical (1970s–2000s) | ARIMA, ETS, Exponential Smoothing |
| ML era (2010s) | Prophet, XGBoost + features |
| Deep Learning (2018–2023) | DeepAR, N-BEATS, PatchTST, TiDE |
| **Foundation Models (2024–)** | **TimesFM, Chronos, Moirai, Moment** |

We focus on **Prophet** (the classic interpretable baseline) and **Chronos** (a zero-shot foundation model).

## 9. Prophet — Decomposition-Based Forecasting

Taylor & Letham (Meta, 2017). Additive decomposition:

$$y(t) = g(t) + s(t) + h(t) + \varepsilon(t)$$

| Term | Meaning |
|------|---------|
| $g(t)$ | Trend (piecewise linear or logistic, automatic changepoints) |
| $s(t)$ | Seasonality (Fourier series — yearly, weekly, daily) |
| $h(t)$ | Holidays / event effects |
| $\varepsilon(t)$ | Noise |

Designed for analysts: handles missing data, outliers, and irregular intervals out of the box.

In [ ]:
from prophet import Prophet

# Generate 3 years of daily sales: trend + yearly + weekly seasonality + noise
n_days = 3 * 365
dates = pd.date_range('2021-01-01', periods=n_days, freq='D')
t = np.arange(n_days)

trend  = 100 + 0.05 * t
yearly = 15 * np.sin(2 * np.pi * t / 365.25)
weekly = 5  * np.sin(2 * np.pi * t / 7)
noise  = np.random.normal(0, 5, n_days)
y = trend + yearly + weekly + noise

df = pd.DataFrame({'ds': dates, 'y': y})
print(f'📅 Generated {len(df)} days of synthetic sales data')

model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05,
)
model.fit(df)

future = model.make_future_dataframe(periods=180)
forecast = model.predict(future)

fig = model.plot(forecast)
plt.title('Prophet Forecast: Sales (3 years history → 6 months ahead)', fontweight='bold')
plt.tight_layout(); plt.show()

fig2 = model.plot_components(forecast)
plt.tight_layout(); plt.show()

print('💡 Prophet decomposes the series into trend + seasonality components — interpretable by design.')

## 10. Foundation Models for Time Series

Why now? Three things came together:

1. **Data availability** — large public TS corpora (Google Trends, weather, finance, Wikipedia)
2. **Transformer architecture** — proven sequence modeller across modalities
3. **Scaling laws** — bigger models + more data = better zero-shot

### Landscape (2024)
| Model | Org | Architecture | Key idea |
|-------|-----|--------------|----------|
| **TimesFM** | Google | Decoder-only Transformer | Patching + 100B+ training points |
| **Chronos** | Amazon | T5 (Enc-Dec) | Tokenize values → language modeling |
| **Moirai** | Salesforce | Masked Encoder | Any-variate, any-frequency, unified |
| **Moment** | CMU | Masked Encoder | Multi-task pretraining on patches |
| **Lag-Llama** | ServiceNow | LLaMA-style | Probabilistic forecasting with lags |

## 11. Chronos — Forecasting as Language Modeling

**Amazon (2024)**. Treat values as tokens, forecast as text generation.

| Step | Description |
|------|-------------|
| **1. Scale** | Normalize by mean absolute value |
| **2. Tokenize** | Quantize into 4096 discrete bins |
| **3. T5 model** | Cross-entropy on token sequences |
| **4. Sample** | Multiple forecast paths → uncertainty |

Available sizes: Mini (20M), Small (46M), Base (200M), Large (710M).
We use **Mini** for speed — works on CPU.

In [ ]:
import torch
from chronos import ChronosPipeline

pipeline = ChronosPipeline.from_pretrained(
    'amazon/chronos-t5-mini',
    device_map='cuda' if torch.cuda.is_available() else 'cpu',
    torch_dtype=torch.float32,
)
print('✅ Chronos-Mini loaded')

context_len = 800
horizon = 60
context = torch.tensor(df['y'].iloc[:context_len].values, dtype=torch.float32)

forecast = pipeline.predict(
    context=context,
    prediction_length=horizon,
    num_samples=100,
)
fc = forecast.numpy().squeeze(0)  # (100 samples, horizon)

low, median, high = np.quantile(fc, [0.1, 0.5, 0.9], axis=0)
fc_dates = df['ds'].iloc[context_len:context_len + horizon].values
actual   = df['y'].iloc[context_len:context_len + horizon].values

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df['ds'].iloc[:context_len], df['y'].iloc[:context_len], color='gray', alpha=0.6, label='Context')
ax.plot(fc_dates, actual, '--', color='black', label='Actual')
ax.plot(fc_dates, median, color='#26B7DB', linewidth=2, label='Chronos median')
ax.fill_between(fc_dates, low, high, color='#26B7DB', alpha=0.25, label='80% interval')
ax.set_title('Chronos: Zero-Shot Probabilistic Forecast', fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Sales'); ax.legend()
plt.tight_layout(); plt.show()

print('💡 No training on this data — Chronos forecasts directly from the pretrained checkpoint.')

## 12. Prophet vs Chronos — Side-by-Side

Same forecasting problem, two paradigms:
- **Prophet**: train per dataset, interpretable components, point + interval forecasts
- **Chronos**: zero-shot, probabilistic (sample paths), no training on this data

In [ ]:
# Fit Prophet on the same context window used for Chronos
df_ctx = df.iloc[:context_len].copy()

m = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
)
m.fit(df_ctx)
future = m.make_future_dataframe(periods=horizon)
prophet_fc = m.predict(future).iloc[-horizon:]

# Compare via MAE
mae_prophet = np.abs(prophet_fc['yhat'].values - actual).mean()
mae_chronos = np.abs(median - actual).mean()

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
for ax, name, fc_vals, color, mae in [
    (axes[0], 'Prophet (fitted)',    prophet_fc['yhat'].values, '#26C281', mae_prophet),
    (axes[1], 'Chronos (zero-shot)', median,                    '#26B7DB', mae_chronos),
]:
    ax.plot(df['ds'].iloc[:context_len], df['y'].iloc[:context_len], color='gray', alpha=0.5, label='Context')
    ax.plot(fc_dates, actual, '--', color='black', label='Actual')
    ax.plot(fc_dates, fc_vals, color=color, linewidth=2, label=name)
    ax.set_title(f'{name}  |  MAE = {mae:.2f}', fontweight='bold')
    ax.legend()
plt.tight_layout(); plt.show()

print(f'\n📊 Prophet MAE: {mae_prophet:.2f}    Chronos MAE: {mae_chronos:.2f}')
print('💡 Prophet trains on this series — interpretable, per-dataset.')
print('   Chronos uses a pretrained model — zero-shot, probabilistic out of the box.')

## 13. Practical Guide — Choosing the Right Tool

### Tabular
| Use this | When... |
|----------|---------|
| **TabPFN** | Small data (< 3K), quick prototyping, no time for HP tuning, classification |
| **XGBoost / LightGBM** | Large data (> 10K), need interpretability (SHAP), production, regression at scale |

### Time Series
| Scenario | Recommended | Why |
|----------|-------------|-----|
| Quick interpretable baseline | **Prophet** | Easy, interpretable, robust |
| Need covariates | Prophet + features (or XGBoost) | Add regressors |
| Zero-shot, new domain | **TimesFM / Chronos** | Pre-trained, no target data needed |
| Uncertainty needed | **Chronos** | Native probabilistic via sampling |
| Multivariate | **Moirai** | Any-variate natively |
| Production + interpretable | **Prophet + XGBoost** | Proven, mature, well-supported |

> **General rule**: start simple (XGBoost for tabular, Prophet for TS),
> validate against foundation models (TabPFN, Chronos), decide based on your constraints.

---
## 📝 Key Takeaways

1. **Tree-based models (XGBoost, LightGBM) remain the gold standard** for most tabular tasks
2. **TabPFN** brings the foundation-model paradigm to tabular data — pretrain on synthetic data, predict in one pass
3. **TabPFN excels on small datasets without tuning**; trees dominate on large-scale data
4. **Prophet** remains a strong, interpretable baseline for time series
5. **TimesFM / Chronos** bring "pre-train once, forecast anything" to time series
6. **Text-to-SQL & Table QA** unlock natural-language access to structured data
7. **The best approach depends on data size, task, and production constraints — there is no universal winner**

### Things to try next
- **TabPFN v2** (2024) — removes the 3K size limit and adds regression
- **NeuralProphet** — Prophet's PyTorch successor with covariate support
- **Moirai** for multivariate forecasting on your own data
- A real **LangChain + ChatOpenAI** Text-to-SQL pipeline against your team's database

---
*Lecture 22 (AI for Data Science) — IMEN574 / IDSC574, Spring 2026*